# `predict_cell_type` tutorial — Basal Ganglia Subclass

End-to-end tutorial for cytozip's methylation cell-type classifier
([`predict_cell_type`](../cytozip/model.py) / [`CellTypeClassifier`](../cytozip/model.py)) on real
**Basal Ganglia (BG)** *Subclass* pseudobulks.

## What the classifier does

Given a **shallow / low-coverage** single cell (few reads per cytosine), assign it to one of a
panel of **deep** cell-type pseudobulk references, using a transparent, training-free
naive-Bayes / methylation-frequency deconvolution:

1. For each candidate type `t` and cytosine `c`, estimate a continuous methylation frequency from
   the pseudobulk with Beta shrinkage $\theta_{c,t} = (m_{c,t}+\alpha_0)/(n_{c,t}+\alpha_0+\beta_0)$.
2. Score a query cell as the aggregated per-cytosine Bernoulli log-likelihood, in two independent
   channels (CpG and CpH).
3. Softmax over types &rarr; calibrated probabilities (supports abstention).

## Pipeline (this notebook covers stage 1 only)

| stage | what | status |
|-------|------|--------|
| **1. Train** | fit per-type CpG/CpH frequencies from the Subclass pseudobulks &rarr; save a reusable model store | **this notebook (run now)** |
| 2. Predict | load the trained model, classify query cells | placeholder — run later |
| 3. Validation | held-out accuracy / macro-F1 / confusion matrix | placeholder — run later |
| 4. Benchmark | speed / memory / downsampling sweeps | placeholder — run later |

> The model is saved so that the later `predict_cell_type(..., outdir=OUTDIR)` call **auto-detects it
> and skips fitting** — no need to re-read the pseudobulks.

## 1. Setup &amp; imports

In [1]:
import os, glob, json, time
import numpy as np
import pandas as pd
import cytozip as czip
from cytozip import CellTypeClassifier, predict_cell_type

print('cytozip', getattr(czip, '__version__', '(dev)'))

cytozip 0.3.9.dev10


## 2. Configuration (inputs, outputs, hyper-parameters)

- **`PSEUDOBULK_DIR`** — the `-s` *source*: a directory of per-cell-type deep pseudobulk `.cz`
  (each file's stem is the cell-type name).
- **`REFERENCE`** — the `-e` *reference*: the `build_ref` allc `.cz` supplying the per-row
  `context` (CpG vs CpH split). All pseudobulks are row-aligned to this axis.
- **`OUTDIR`** — where everything (model now, predictions later) is written. The model store goes
  to `OUTDIR/model` (real disk, so a large memmap never spills into a small/RAM `/tmp` &rarr; SIGBUS).

In [2]:
# --- inputs -------------------------------------------------------------
PSEUDOBULK_DIR = os.path.expanduser('~/Projects/BG/pseudobulk/Subclass/cz')     # -s source (per-type .cz)
REFERENCE      = os.path.expanduser('~/Ref/hg38/hg38_with_chrL.allc.cz')         # -e reference (context axis)

# --- outputs ------------------------------------------------------------
OUTDIR    = os.path.expanduser('~/Projects/BG/pseudobulk/Subclass/model')        # model + later predictions land here
MODEL_DIR = os.path.join(OUTDIR, 'model')                                        # trained model store (reused by predict)
os.makedirs(OUTDIR, exist_ok=True)

# --- training hyper-parameters -----------------------------------------
LAMBDA_CG = 1.0    # CpG channel log-weight
LAMBDA_CH = 1.0    # CpH channel log-weight (lower it to rebalance if CpH sites dominate)
TOP_CG    = 0.10   # keep top 10% most discriminative CpG sites (None = keep all)
TOP_CH    = 0.05   # keep top 5%  most discriminative CpH sites (None = keep all)
N_JOBS    = 8      # parallel .cz readers during fit

for p in (PSEUDOBULK_DIR, REFERENCE):
    assert os.path.exists(p), f'missing input: {p}'
print('pseudobulk dir :', PSEUDOBULK_DIR)
print('reference      :', REFERENCE)
print('output dir     :', OUTDIR)
print('model store    :', MODEL_DIR)

pseudobulk dir : /home/x-wding2/Projects/BG/pseudobulk/Subclass/cz
reference      : /home/x-wding2/Ref/hg38/hg38_with_chrL.allc.cz
output dir     : /home/x-wding2/Projects/BG/pseudobulk/Subclass/model
model store    : /home/x-wding2/Projects/BG/pseudobulk/Subclass/model/model


## 3. Inspect the pseudobulk inputs

`fit` now accepts the **directory** directly (`pseudobulks=PSEUDOBULK_DIR`) — each `.cz` file's stem
becomes the cell-type label, and non-`.cz` files are ignored. Here we still build the explicit
`{cell_type: path}` mapping just to **preview** the inputs and to align the `cell_counts` below.


In [3]:
cz_files = sorted(glob.glob(os.path.join(PSEUDOBULK_DIR, '*.cz')))
pseudobulks = {os.path.basename(p)[:-3]: p for p in cz_files}   # stem -> path

print(f'{len(pseudobulks)} cell types found:')
for t, p in pseudobulks.items():
    print(f'  {t:<24s} {os.path.getsize(p)/1e6:8.1f} MB')

32 cell types found:
  ACx_MEIS2_GABA              545.5 MB
  Astrocyte                  2411.0 MB
  CN_Cholinergic_GABA        1090.7 MB
  CN_GABA-Glut                787.6 MB
  CN_LAMP5-CXCL14_GABA       1602.1 MB
  CN_LAMP5-LHX6_GABA          950.8 MB
  CN_LHX8_GABA               1550.3 MB
  CN_MEIS2_GABA              1533.7 MB
  CN_ONECUT1_GABA            1064.8 MB
  CN_ST18_GABA               2764.8 MB
  CN_VIP_GABA                1685.8 MB
  Endo                       1199.0 MB
  F_GABA                     2555.0 MB
  F_Glut                     1814.1 MB
  F_M_GATA3_GABA             2737.0 MB
  F_M_Glut                   2466.5 MB
  Glut                       2053.7 MB
  Lymphocyte                  901.9 MB
  M_Dopa                     1243.5 MB
  Microglia                  2181.0 MB
  OPC                        1822.0 MB
  OT_Granular_GABA           1260.1 MB
  Oligodendrocyte            3265.0 MB
  Pericyte                   1200.3 MB
  SMC                         635.7 MB
  SN

## 4. (Optional) abundance prior — `cell_counts`

`cell_counts` records how many cells belong to each Subclass. It does **not** affect the fitted
frequencies; it is stored on the model and only used at **predict** time when `prior_alpha > 0`
(types weighted by $\pi_t \propto \text{cell\_counts}[t]^{\text{prior\_alpha}}$).

The true per-Subclass cell counts come from the atlas annotation table
`~/Projects/BG/clustering/100kb/annotations.tsv` — we take `value_counts()` on its **`Subclass`**
column. Annotation names use spaces while the `.cz` stems use underscores, so we normalise spaces
→ underscores before aligning to our cell types. Prediction still defaults to a **uniform** prior
(`prior_alpha=0`); set `USE_ABUNDANCE_PRIOR = True` to bake the counts into the model.


In [ ]:
USE_ABUNDANCE_PRIOR = False   # keep uniform prior by default; set True to bake in the counts

# True per-Subclass cell counts from the atlas annotation table.
ANNOT_TSV = os.path.expanduser('~/Projects/BG/clustering/100kb/annotations.tsv')

vc = pd.read_csv(ANNOT_TSV, sep='\t', usecols=['Subclass'])['Subclass'].value_counts()
counts_all = {str(name).replace(' ', '_'): int(n) for name, n in vc.items()}   # spaces -> underscores

cell_counts = {t: counts_all[t] for t in pseudobulks if t in counts_all}
missing = [t for t in pseudobulks if t not in counts_all]
print(f'{len(vc)} Subclasses in annotations; matched {len(cell_counts)}/{len(pseudobulks)} pseudobulk types')
if missing:
    print('unmatched pseudobulk types:', missing)
print('example counts:', dict(list(cell_counts.items())[:5]))


matched 32/32 types; missing: []


## 5. Train — fit the classifier and save the model store

`fit` reads every pseudobulk `.cz` once, estimates the per-type CpG/CpH frequencies, keeps the
`top_cg` / `top_ch` most discriminative sites, and spills the `(n_sites x n_types)` log-likelihood
tables to memory-mappable `.npy` files. Passing `outdir=MODEL_DIR` routes that store **straight to
real disk**; `save(MODEL_DIR)` then writes the `meta.json` header.

This is the same fitting `predict_cell_type` does internally — pre-computing it here means the later
predict/validation/benchmark calls just **load** this model and skip fitting.

In [ ]:
t0 = time.time()
clf = CellTypeClassifier(lambda_cg=LAMBDA_CG, lambda_ch=LAMBDA_CH).fit(
    pseudobulks=PSEUDOBULK_DIR,                            # pass the directory directly (stem = cell type)
    reference=REFERENCE,                                   # supplies per-row context (CpG/CpH split)
    cell_counts=(cell_counts if USE_ABUNDANCE_PRIOR else None),
    top_cg=TOP_CG, top_ch=TOP_CH,                          # keep only discriminative sites
    n_jobs=N_JOBS,
    outdir=MODEL_DIR,                                      # memmap store -> real disk (avoids /tmp SIGBUS)
)
clf.save(MODEL_DIR)                                        # write meta.json (arrays already in MODEL_DIR)
print(f'trained {len(clf.cell_types)} cell types in {time.time() - t0:.1f}s')
print('model saved to', MODEL_DIR)


KeyboardInterrupt: 

## 6. Inspect the trained model

Confirm what `fit` selected and the on-disk footprint of the reusable store.

In [ ]:
meta = json.load(open(os.path.join(MODEL_DIR, 'meta.json')))
print('cell types      :', len(meta['cell_types']))
print('CpG sites kept  :', meta['cg']['n_sites'] if meta['cg'] else 0)
print('CpH sites kept  :', meta['ch']['n_sites'] if meta['ch'] else 0)
print('alpha0/beta0 CG :', meta['alpha0_cg'], '/', meta['beta0_cg'])
print('alpha0/beta0 CH :', meta['alpha0_ch'], '/', meta['beta0_ch'])

store_size = sum(os.path.getsize(os.path.join(MODEL_DIR, f)) for f in os.listdir(MODEL_DIR))
print('model store size: %.1f MB' % (store_size / 1e6))
print('files           :', sorted(os.listdir(MODEL_DIR)))

---
## 7. NEXT — Predict (run later)

**Do not run yet.** Once the query cells are ready, `predict_cell_type` will detect the trained model
under `OUTDIR/model` and **skip fitting**, then write `predictions.csv` + `predict_proba.csv` to
`OUTDIR`. `query` can be a directory of single-cell `.cz`, a concatenated multi-cell `.cz`, a
`{cell_id: path}` dict, or a 2-column `[cell_id, cz_path]` table.

```python
labels, proba = predict_cell_type(
    query=QUERY,               # <- to be provided
    outdir=OUTDIR,             # reuses OUTDIR/model, writes predictions.csv + predict_proba.csv
    reference=REFERENCE,       # only used if the model has to be re-fit
    prior_alpha=0.0,           # uniform prior; raise to use the abundance prior
    abstain_threshold=None,    # e.g. 0.6 to abstain on low-confidence cells
    n_jobs=N_JOBS,
)
```

## 8. NEXT — Validation (run later)

**Do not run yet.** Evaluate the classifier on cells with known labels (e.g. held-out single cells or
downsampled pseudobulks): overall accuracy, **macro-F1** (fair to rare types), and the confusion
matrix. To be filled in when the labelled query set is provided.

## 9. NEXT — Benchmark (run later)

**Do not run yet.** Measure runtime / peak memory and sweep robustness knobs
(`max_query_cg` / `max_query_ch` downsampling, `top_cg` / `top_ch`, `lambda_ch`, `prior_alpha`)
against accuracy. To be filled in after validation.